This colab was created for
https://github.com/minipasila/omnivoice-api-server

**Now updated to use OmniVoice!**
Maybe you wanted to try OmniVoice but your computer doesn't allow it, then this colab will help you.

If you liked colab and the xtts-api-server project, give it a star

In [ ]:
#@markdown #####You can enable gdrive that drag and drop samples directly through your google drive via the `drive/Mydrive` path
mount_gdrive = False #@param{type:"boolean"}
#@markdown #####Enter the GitHub URL of your modified repository (defaults to the main repo)
repo_url = "https://github.com/minipasila/omnivoice-api-server.git" #@param{type:"string"}

if mount_gdrive:
  from google.colab import drive
  drive.mount('/kaggle/working/drive')

print("Installing System Dependencies (PortAudio)...\nPlease wait...")
!apt-get update > '/dev/null' 2>&1
!apt-get install -y portaudio19-dev > '/dev/null' 2>&1

print("Installing Packages and Dependencies...\nPlease wait 2-3 minutes")
!npm install localtunnel > '/dev/null' 2>&1

# Clone the modified repository
!rm -rf /kaggle/working/omnivoice-api-server
!git clone {repo_url} /kaggle/working/omnivoice-api-server > '/dev/null' 2>&1
%cd /kaggle/working/omnivoice-api-server

# Install PyTorch and OmniVoice requirements
!pip -q install torch>=2.4 torchaudio>=2.4 > '/dev/null' 2>&1
!pip -q install -r requirements.txt > '/dev/null' 2>&1
!pip -q install . > '/dev/null' 2>&1

print("Downloading a test sample")
!mkdir -p speakers > '/dev/null' 2>&1
!wget -q -O speakers/example.wav https://raw.githubusercontent.com/minipasila/omnivoice-api-server/refs/heads/main/example/female.wav

import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

print("Done!")

### 🎤 How to add your own Custom Voices

To use your own voice samples, you need to add them to this Kaggle notebook as a "Dataset".
1. Look at the right-hand sidebar in Kaggle.
2. Click on **"Upload"** (next to "+ Add Input").
3. Click **"New Dataset"** and upload a folder containing your short `.wav` voice samples.
4. Give your dataset a name and click **Create**.

Once your dataset finishes uploading and processing, simply **run the cell below**. It will automatically find your audio files and copy them into the correct `speakers` folder!

In [ ]:
import os
import shutil

# Define directories
speaker_dir = '/kaggle/working/omnivoice-api-server/speakers/'
input_dir = '/kaggle/input/'

# Make sure the speakers directory exists
os.makedirs(speaker_dir, exist_ok=True)

found_audio = False
supported_formats = ('.wav', '.mp3', '.flac', '.ogg')

print("Scanning Kaggle inputs for voice samples...\n")

# Walk through all attached datasets and folders
for root, dirs, files in os.walk(input_dir):
    for file in files:
        if file.lower().endswith(supported_formats):
            source_path = os.path.join(root, file)
            dest_path = os.path.join(speaker_dir, file)
            
            # Copy the file to the speakers folder
            shutil.copy2(source_path, dest_path)
            print(f"✅ Found and copied: {file}")
            found_audio = True

print("\n" + "-"*30)
if not found_audio:
    print("⚠️ No audio files found! Make sure you uploaded your dataset and added it to the notebook.")
else:
    print("🎉 All custom voices loaded successfully!")

In [ ]:
#@markdown A link and ip address will appear in front of you. You need to follow the link and paste there the given ip address.
#@markdown
#@markdown After that you can copy the link and paste it into SillyTavern.
import time
import os
import subprocess

# Ensure we are in the right directory
os.chdir('/kaggle/working/omnivoice-api-server')

# Run localtunnel in the background using subprocess instead of the ! command
subprocess.Popen('npx lt -p 8020 > lt.log 2>&1', shell=True)

time.sleep(2)
with open('lt.log', 'r') as testwritefile:
    tunnel_ip = testwritefile.read()
    print(tunnel_ip)
    print("Before you copy the link above click on it and copy that ip:")
    !curl ipv4.icanhazip.com
    print("\n\n\n\n")

# Start the API server
!python -m xtts_api_server -hs 0.0.0.0